# TennisClip CV 增强调试工作流 · Kaggle GPU 验证

把 `backend/app` 完整移植到 Kaggle 免费 GPU（P100/T4）上，跑「CV 增强调试工作流」
（`detect.tracknet` → `post.score_highlights` → `edit.concat` / `report.technical` /
`post.visualize_track`），产物落 `/kaggle/working/` 供下载人工核对（对齐方案 14 §2.4）。

## 运行前设置（Notebook Settings）

| 项 | 值 | 说明 |
|---|---|---|
| Input Data | `{username}/kaggle-tennisclip-verify` | 勾选本 notebook 对应的 dataset |
| Accelerator | **GPU T4 16GB / P100** | 必须，TrackNet 逐帧推理需 GPU |
| Internet | **ON** | CLIP 需下载 HF 权重；GPU 配额按开联网计 |
| 运行时长上限 | 9h（默认） | 单条视频远不会触顶 |

## 输入 dataset 布局（`/kaggle/input/{slug}/`，平铺：kaggle CLI 2.x 默认 dir_mode=skip 子目录不上传）

```
verify.py             # 验证入口（开跑前自动规整：根下 mp4 → data/、根下 *.pth → weights/）
app.tar.gz            # backend/{app,prompts} 打包（解压到 /kaggle/working/）
dataset_sample.mp4  # 待验证视频（1-5 分钟，720p/30fps；可多放，文件名避开 dataset_sample.mp4）
tracknet.pth        # TrackNet 权重（根下平铺，可多个 *.pth）
```

In [ ]:
# 1. 安装依赖（loguru/pydantic/pyyaml/python-dotenv 随 app 包；torch/cv2/ultralytics/transformers pip 装）
!pip install -q loguru pydantic pyyaml python-dotenv numpy
!pip install -q opencv-python-headless
!pip install -q torch torchvision 2>&1 | tail -1
!pip install -q ultralytics 2>&1 | tail -1
!pip install -q transformers 2>&1 | tail -1
# Kaggle 侧关 loguru 文件 sink（避免 /kaggle/working 之外路径写失败）
import os
os.environ["TENNISCLIP_LOG_ENQUEUE"] = "0"

In [ ]:
# 2. 解压 backend/{app,prompts}（dataset 根下的 app.tar.gz 或 app.zip）
#    解压到 /kaggle/working/，--repo 指向该根（含 app/ + prompts/ 两个顶层包）
import os, subprocess

OWNER = "chikshadaileng"          # 你的 Kaggle 用户名
SLUG = "kaggle-tennisclip-verify"   # dataset slug
IN = f"/kaggle/input/{SLUG}"
WORK = "/kaggle/working"
os.environ["TENNISCLIP_LOG_ENQUEUE"] = "0"   # Kaggle 侧关 loguru enqueue（避免写 /kaggle 之外路径失败）

# —— 兜底：Input Data 未挂载时（kaggle CLI 2.x 的 dataset_sources 可能未注册成 Input Data），
#    用 Kaggle 预装 kaggle CLI 直接下载 dataset 到 /kaggle/working/ds/，再用它做输入
if not os.path.isdir(IN):
    print(f"[i] {IN} 不存在（Input Data 未挂载）→ 自动 kaggle datasets download {OWNER}/{SLUG}")
    IN = os.path.join(WORK, "ds")
    os.makedirs(IN, exist_ok=True)
    # Kaggle 预装 kaggle CLI 2.0.2（比本地 2.2.x 旧）：-f/--file 需带参数，多文件 dataset 不用 -f；
    # --unzip 直接解压成平铺文件（不产生中间 zip）；-o 覆盖旧下载
    subprocess.run(["kaggle", "datasets", "download", f"{OWNER}/{SLUG}",
                   "-p", IN, "--unzip", "-o"], check=True)
    print(f"dataset 已下载到 {IN}：", sorted(os.listdir(IN)))

# 诊断：打印 dataset 实际内容（缺 app.tar.gz 时能看到当前版本里到底有什么）
print("dataset 内容：", sorted(os.listdir(IN)) if os.path.isdir(IN) else f"[缺失] {IN} 不存在")

# 解压 app 包（.tar.gz 或 .zip，build_dataset.ps1 二选一产出；含 app/ + prompts/）
tar = f"{IN}/app.tar.gz"
zip = f"{IN}/app.zip"
if os.path.exists(tar):
    subprocess.run(["tar", "-xzf", tar, "-C", WORK], check=True)
    print(f"解压 {tar} → {WORK}/app/ + prompts/")
elif os.path.exists(zip):
    subprocess.run(["unzip", "-o", zip, "-d", WORK], check=True)
    print(f"解压 {zip} → {WORK}/app/ + prompts/")
else:
    raise FileNotFoundError(
        f"dataset 缺 app.tar.gz / app.zip：{IN}\n"
        f"当前内容：{sorted(os.listdir(IN)) if os.path.isdir(IN) else '[目录不存在]'}\n"
        f"排查：本地 kaggle/dataset/ 是否有 app.tar.gz（python deploy.py --skip-upload 重组装）；"
        f"Kaggle 上 dataset 版本是否含 app.tar.gz（kaggle datasets list 看版本）")

print("repo 根内容：", sorted(os.listdir(WORK)))
assert os.path.isdir(f"{WORK}/app"), "app/ 未解压成功"
assert os.path.isdir(f"{WORK}/prompts"), "prompts/ 未解压成功"

In [ ]:
# 3. 跑验证（完整 16 节点 executor 图，GPU 逐帧推理）
# 视频 = dataset 根下全部 mp4（Kaggle 上传后平铺在根下，CLI 默认 dir_mode=skip）
# 产物落 /kaggle/working/kaggle-XX/ + summary.json
import subprocess, sys
cmd = [
    sys.executable, f"{IN}/verify.py",
    "--repo", WORK,
    "--input", IN,
    "--out", WORK,
    "--device", "cuda",
]
print("CMD:", " ".join(cmd))
subprocess.run(cmd, check=False)

In [ ]:
# 4. 列出产物（Notebook OUTPUT 区可见，供下载）
!find /kaggle/working -name 'track_overlay.mp4' -o -name 'result.json' -o -name 'summary.json' 2>/dev/null | sort

## 人工核对（对齐方案 14 §2.4 可视化验证）

运行完成后，在 Notebook 下方 **OUTPUT** 区下载：

- `kaggle-XX/track_overlay.mp4` — 播放逐回合核对：
  - **G1**：金标回合内球轨迹是否连续（红色未检出帧 = 漏检）
  - **G2**：静止段（灰标 `static?`）是否落在金标「回合外」（球桶/捡球区）；若与某回合重叠 → 死球误入
  - **多球选球**：双球同屏时，被后续回合引用的轨迹段是否对应「活球」
  - **捡球间隙**：时间线红色段是否覆盖金标回合间静止区间（静止 ≥1.5s 结束规则）
- `kaggle-XX/result.json` — 节点进度 + 高光段 + 产物清单
- `summary.json` — 全部视频汇总

多球跳变明显时（TrackNet argmax 单点帧间跳变），调大 `post.visualize_track` 的 `max_speed` 参数
（镜头特写球场占画面比例大时）或调小 `break_gap`，重跑核对直至 G2 死球误入率清零。